In [ ]:
# === SUPERVISED PATTERN CLASSIFIER ===
# Trains a CNN to predict future price movements from candlestick patterns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

# Config
LOOKBACK = 288
BATCH_SIZE = 256
EPOCHS = 20
LR = 3e-4
DATA_PATH = 'data/binance-BTCUSDT-5m.pkl'

# Load data
df = pd.read_pickle(DATA_PATH)
print(f"Loaded {len(df):,} samples")

# Label generation: Future return (next N candles)
FUTURE_WINDOW = 12  # 1 hour ahead (12 * 5min)
df['future_return'] = df['close'].pct_change(FUTURE_WINDOW).shift(-FUTURE_WINDOW)

# Labels: 0=DOWN, 1=NEUTRAL, 2=UP
THRESHOLD = 0.002  # 0.2% threshold
df['label'] = 1  # Default neutral
df.loc[df['future_return'] > THRESHOLD, 'label'] = 2  # Up
df.loc[df['future_return'] < -THRESHOLD, 'label'] = 0  # Down
df = df.dropna()

# Features: OHLCV normalized
features = ['open', 'high', 'low', 'close', 'volume']
for col in features:
    df[f'{col}_norm'] = (df[col] - df[col].rolling(LOOKBACK).mean()) / df[col].rolling(LOOKBACK).std()
df = df.dropna()

# Dataset
class PatternDataset(Dataset):
    def __init__(self, df, lookback):
        self.df = df.reset_index(drop=True)
        self.lookback = lookback
        self.feature_cols = [f'{c}_norm' for c in features]
        
    def __len__(self):
        return len(self.df) - self.lookback
    
    def __getitem__(self, idx):
        start = idx
        end = idx + self.lookback
        X = self.df.loc[start:end-1, self.feature_cols].values.astype(np.float32)
        y = self.df.loc[end-1, 'label']
        return torch.from_numpy(X), torch.tensor(y, dtype=torch.long)

# Split
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:]

train_ds = PatternDataset(train_df, LOOKBACK)
val_ds = PatternDataset(val_df, LOOKBACK)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

# Model: Multi-scale CNN
class PatternCNN(nn.Module):
    def __init__(self, input_dim=5, hidden=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=10, padding=5),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3)  # 3 classes
        )
    
    def forward(self, x):
        x = x.transpose(1, 2)  # [B, T, C] -> [B, C, T]
        x = self.cnn(x).squeeze(-1)  # [B, 64]
        return self.fc(x)

model = PatternCNN().cuda()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Training
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for X, y in train_loader:
        X, y = X.cuda(), y.cuda()
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.cuda(), y.cuda()
            out = model(X)
            val_loss += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
    
    acc = correct / len(val_ds) * 100
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Acc: {acc:.2f}%")

# Save
torch.save(model.state_dict(), 'pattern_classifier.pth')
print("✓ Model saved")

# Inference example
model.eval()
with torch.no_grad():
    sample_X, sample_y = val_ds[0]
    pred = model(sample_X.unsqueeze(0).cuda())
    probs = torch.softmax(pred, dim=1)[0]
    print(f"\nPrediction: DOWN={probs[0]:.2%} | NEUTRAL={probs[1]:.2%} | UP={probs[2]:.2%}")
    print(f"Actual: {['DOWN', 'NEUTRAL', 'UP'][sample_y]}")


In [ ]:
# Save
torch.save(model.state_dict(), 'pattern_classifier.pth')
print("✓ Model saved")

# Inference example
model.eval()
with torch.no_grad():
    sample_X, sample_y = val_ds[0]
    pred = model(sample_X.unsqueeze(0).cuda())
    probs = torch.softmax(pred, dim=1)[0]
    print(f"\nPrediction: DOWN={probs[0]:.2%} | NEUTRAL={probs[1]:.2%} | UP={probs[2]:.2%}")
    print(f"Actual: {['DOWN', 'NEUTRAL', 'UP'][sample_y]}")